In [0]:
-- ============================================================================
-- MegaMart Databricks Capstone
-- Stage: 05 - Inventory Analysis
--
-- Purpose:
--   Analyze inventory position, sales velocity, and supplier delivery
--   performance using the cleaned Silver layer.
--
-- Source:
--   megamart_dev.silver.*
--
-- Target:
--   megamart_dev.gold.*
--
-- Generator-specific decisions:
--   1. Inventory contains one snapshot per store/product combination.
--   2. Products contain supplier_id, so product -> supplier is available.
--   3. Purchase orders do not contain product_id or quantity_ordered.
--   4. Purchase orders do not contain a promised delivery date/SLA.
--
-- Therefore:
--   - stockout/low-stock analysis is based on current quantity_on_hand.
--   - sales velocity is based on historical sales quantity.
--   - supplier lead time is based on delivered purchase orders.
--   - delivery completion rate is used instead of inventing an on-time SLA.
--   - reorder-point calculation is handled by the following Python stage.
-- ============================================================================


-- ============================================================================
-- 1. Select Project Catalog
-- ============================================================================

USE CATALOG megamart_dev;


-- ============================================================================
-- 2. Source Data Verification
-- ============================================================================

SELECT
    'inventory' AS dataset,
    COUNT(*) AS record_count
FROM silver.inventory

UNION ALL

SELECT
    'sales_transactions' AS dataset,
    COUNT(*) AS record_count
FROM silver.sales_transactions

UNION ALL

SELECT
    'products' AS dataset,
    COUNT(*) AS record_count
FROM silver.products

UNION ALL

SELECT
    'suppliers' AS dataset,
    COUNT(*) AS record_count
FROM silver.suppliers

UNION ALL

SELECT
    'purchase_orders' AS dataset,
    COUNT(*) AS record_count
FROM silver.purchase_orders;


-- ============================================================================
-- 3. Inventory Status
--
-- Answers:
--   - Which products are out of stock?
--   - Which products are low stock?
--
-- The project uses the existing configuration/business requirement
-- of LOW STOCK < 50 units.
--
-- Zero inventory is NOT treated as a data-quality error.
-- It is a valid business condition and is retained.
-- ============================================================================

CREATE OR REPLACE TABLE gold.inventory_status
USING DELTA
AS
SELECT
    i.inventory_id,

    i.store_id,
    s.store_name,
    s.location,

    i.product_id,
    p.product_name,
    p.category,

    p.supplier_id,
    sup.supplier_name,

    i.quantity_on_hand,

    CASE
        WHEN i.quantity_on_hand = 0
            THEN 'OUT_OF_STOCK'

        WHEN i.quantity_on_hand < 50
            THEN 'LOW_STOCK'

        ELSE 'HEALTHY'
    END AS stock_status,

    CASE
        WHEN i.quantity_on_hand = 0
            THEN 3

        WHEN i.quantity_on_hand < 50
            THEN 2

        ELSE 1
    END AS stock_priority,

    i._source_file,
    i._ingestion_timestamp

FROM silver.inventory i

INNER JOIN silver.stores s
    ON i.store_id = s.store_id

INNER JOIN silver.products p
    ON i.product_id = p.product_id

LEFT JOIN silver.suppliers sup
    ON p.supplier_id = sup.supplier_id;


-- ============================================================================
-- 4. Product / Store Sales Velocity
--
-- Sales velocity is calculated at store + product level because inventory
-- is also tracked at store + product level.
--
-- Formula:
--
--   average_daily_sales =
--       total_quantity_sold / number_of_days_in_2024
--
-- 2024 is a leap year, so the calendar period contains 366 days.
--
-- We also calculate:
--   - transaction count
--   - total revenue
--   - velocity rank
--   - stock status
--
-- This creates a useful base for the Python reorder optimizer.
-- ============================================================================

CREATE OR REPLACE TABLE gold.product_velocity
USING DELTA
AS
WITH sales_by_store_product AS (

    SELECT
        store_id,
        product_id,

        SUM(quantity_sold)
            AS total_quantity_sold,

        COUNT(DISTINCT transaction_id)
            AS transaction_count,

        SUM(total_amount)
            AS total_revenue

    FROM silver.sales_transactions

    GROUP BY
        store_id,
        product_id
),

combined AS (

    SELECT
        i.inventory_id,

        i.store_id,
        s.store_name,
        s.location,

        i.product_id,
        p.product_name,
        p.category,

        p.supplier_id,
        sup.supplier_name,

        i.quantity_on_hand,

        COALESCE(
            sbsp.total_quantity_sold,
            0
        ) AS total_quantity_sold,

        COALESCE(
            sbsp.transaction_count,
            0
        ) AS transaction_count,

        COALESCE(
            sbsp.total_revenue,
            0
        ) AS total_revenue

    FROM silver.inventory i

    INNER JOIN silver.stores s
        ON i.store_id = s.store_id

    INNER JOIN silver.products p
        ON i.product_id = p.product_id

    LEFT JOIN silver.suppliers sup
        ON p.supplier_id = sup.supplier_id

    LEFT JOIN sales_by_store_product sbsp
        ON i.store_id = sbsp.store_id
        AND i.product_id = sbsp.product_id
)

SELECT
    inventory_id,

    store_id,
    store_name,
    location,

    product_id,
    product_name,
    category,

    supplier_id,
    supplier_name,

    quantity_on_hand,

    total_quantity_sold,
    transaction_count,

    ROUND(
        total_revenue,
        2
    ) AS total_revenue,

    ROUND(
        total_quantity_sold / 366.0,
        4
    ) AS average_daily_sales,

    ROUND(
        CASE
            WHEN quantity_on_hand > 0
            THEN quantity_on_hand
                 / NULLIF(
                     total_quantity_sold / 366.0,
                     0
                 )
            ELSE 0
        END,
        2
    ) AS estimated_days_of_inventory,

    DENSE_RANK() OVER (
        ORDER BY
            total_quantity_sold DESC,
            product_id,
            store_id
    ) AS overall_velocity_rank,

    DENSE_RANK() OVER (
        PARTITION BY store_id
        ORDER BY
            total_quantity_sold DESC,
            product_id
    ) AS store_velocity_rank,

    CASE
        WHEN total_quantity_sold = 0
            THEN 'NO_SALES'

        WHEN PERCENT_RANK() OVER (
            ORDER BY total_quantity_sold
        ) >= 0.80
            THEN 'FAST_MOVING'

        WHEN PERCENT_RANK() OVER (
            ORDER BY total_quantity_sold
        ) <= 0.20
            THEN 'SLOW_MOVING'

        ELSE 'NORMAL_MOVING'
    END AS velocity_class,

    CASE
        WHEN quantity_on_hand = 0
            THEN 'OUT_OF_STOCK'

        WHEN quantity_on_hand < 50
            THEN 'LOW_STOCK'

        ELSE 'HEALTHY'
    END AS stock_status

FROM combined;


-- ============================================================================
-- 5. Slow-Moving / Overstock Candidates
--
-- This view/table is represented inside product_velocity.
--
-- A candidate is:
--
--   SLOW_MOVING
--       AND
--   quantity_on_hand >= 50
--
-- This means the product has relatively low sales velocity while still
-- carrying at least the project's healthy-stock threshold.
--
-- This is a screening rule, not a claim that every such item is definitely
-- overstocked.
-- ============================================================================


-- ============================================================================
-- 6. Supplier Delivery Performance
--
-- PRD requirement:
--   Supplier reliability based on delivery performance.
--
-- Generator behavior:
--   - delivery_date populated  -> delivered order
--   - delivery_date blank/NULL -> incomplete/pending order
--
-- Project definition:
--
--   delivery_reliability_pct =
--       delivered_orders / total_purchase_orders * 100
--
-- Therefore, purchase orders with no delivery_date count against
-- supplier delivery reliability.
--
-- IMPORTANT:
--   This is a delivery reliability/completion metric.
--   It is not a strict SLA-based "on-time delivery rate", because
--   the generator does not provide a promised delivery date.
-- ============================================================================

CREATE OR REPLACE TABLE gold.supplier_scorecard
USING DELTA
AS
SELECT
    sup.supplier_id,
    sup.supplier_name,

    COUNT(po.po_id) AS total_purchase_orders,

    COUNT(
        CASE
            WHEN po.delivery_date IS NOT NULL
            THEN po.po_id
        END
    ) AS delivered_orders,

    COUNT(
        CASE
            WHEN po.delivery_date IS NULL
            THEN po.po_id
        END
    ) AS failed_or_incomplete_orders,

    ROUND(
        (
            COUNT(
                CASE
                    WHEN po.delivery_date IS NOT NULL
                    THEN po.po_id
                END
            )
            / NULLIF(
                COUNT(po.po_id),
                0
            )
        ) * 100,
        2
    ) AS delivery_reliability_pct,

    ROUND(
        AVG(
            CASE
                WHEN po.delivery_date IS NOT NULL
                THEN po.lead_time_days
            END
        ),
        2
    ) AS average_lead_time_days,

    MIN(
        CASE
            WHEN po.delivery_date IS NOT NULL
            THEN po.lead_time_days
        END
    ) AS minimum_lead_time_days,

    MAX(
        CASE
            WHEN po.delivery_date IS NOT NULL
            THEN po.lead_time_days
        END
    ) AS maximum_lead_time_days,

    'RELIABILITY_BASED_ON_DELIVERED_ORDERS'
        AS reliability_definition

FROM silver.suppliers sup

LEFT JOIN silver.purchase_orders po
    ON sup.supplier_id = po.supplier_id

GROUP BY
    sup.supplier_id,
    sup.supplier_name;


-- ============================================================================
-- 7. Inventory Analysis Verification
-- ============================================================================

SELECT
    'inventory_status' AS table_name,
    COUNT(*) AS record_count
FROM gold.inventory_status

UNION ALL

SELECT
    'product_velocity' AS table_name,
    COUNT(*) AS record_count
FROM gold.product_velocity

UNION ALL

SELECT
    'supplier_scorecard' AS table_name,
    COUNT(*) AS record_count
FROM gold.supplier_scorecard;


-- ============================================================================
-- 8. Stock Status Summary
-- ============================================================================

SELECT
    stock_status,
    COUNT(*) AS store_product_count

FROM gold.inventory_status

GROUP BY
    stock_status

ORDER BY
    stock_status;


-- ============================================================================
-- 9. Velocity Summary
-- ============================================================================

SELECT
    velocity_class,
    COUNT(*) AS store_product_count

FROM gold.product_velocity

GROUP BY
    velocity_class

ORDER BY
    velocity_class;


-- ============================================================================
-- 10. Out-of-Stock and Low-Stock Products
-- ============================================================================

SELECT
    store_id,
    store_name,
    product_id,
    product_name,
    category,
    quantity_on_hand,
    stock_status,
    supplier_id,
    supplier_name

FROM gold.inventory_status

WHERE stock_status IN (
    'OUT_OF_STOCK',
    'LOW_STOCK'
)

ORDER BY
    stock_priority DESC,
    quantity_on_hand ASC,
    store_id,
    product_id;


-- ============================================================================
-- 11. Fast-Moving Products
-- ============================================================================

SELECT
    overall_velocity_rank,
    store_id,
    store_name,
    product_id,
    product_name,
    category,
    total_quantity_sold,
    average_daily_sales,
    quantity_on_hand,
    estimated_days_of_inventory,
    velocity_class

FROM gold.product_velocity

WHERE velocity_class = 'FAST_MOVING'

ORDER BY
    overall_velocity_rank;


-- ============================================================================
-- 12. Slow-Moving / Potential Overstock Candidates
-- ============================================================================

SELECT
    store_id,
    store_name,
    product_id,
    product_name,
    category,
    total_quantity_sold,
    average_daily_sales,
    quantity_on_hand,
    estimated_days_of_inventory,
    velocity_class

FROM gold.product_velocity

WHERE velocity_class = 'SLOW_MOVING'
  AND quantity_on_hand >= 50

ORDER BY
    average_daily_sales ASC,
    quantity_on_hand DESC;


-- ============================================================================
-- 13. Supplier Performance
-- ============================================================================

SELECT
    supplier_id,
    supplier_name,
    total_purchase_orders,
    delivered_orders,
    failed_or_incomplete_orders,
    delivery_reliability_pct,
    average_lead_time_days,
    minimum_lead_time_days,
    maximum_lead_time_days,
    reliability_definition

FROM gold.supplier_scorecard

ORDER BY
    delivery_reliability_pct DESC,
    average_lead_time_days ASC,
    supplier_id;
    

-- ============================================================================
-- 14. Store-Level Inventory Summary
-- ============================================================================

SELECT
    store_id,
    store_name,

    COUNT(*) AS total_products,

    SUM(
        CASE
            WHEN stock_status = 'OUT_OF_STOCK'
            THEN 1
            ELSE 0
        END
    ) AS out_of_stock_products,

    SUM(
        CASE
            WHEN stock_status = 'LOW_STOCK'
            THEN 1
            ELSE 0
        END
    ) AS low_stock_products,

    SUM(
        CASE
            WHEN stock_status = 'HEALTHY'
            THEN 1
            ELSE 0
        END
    ) AS healthy_stock_products,

    SUM(quantity_on_hand)
        AS total_quantity_on_hand

FROM gold.inventory_status

GROUP BY
    store_id,
    store_name

ORDER BY
    store_id;


-- ============================================================================
-- 15. Audit Pipeline Run
-- ============================================================================

INSERT INTO audit.pipeline_runs
SELECT
    CONCAT(
        '05_inventory_',
        DATE_FORMAT(
            CURRENT_TIMESTAMP(),
            'yyyyMMddHHmmss'
        )
    ) AS run_id,

    'megamart_pipeline' AS pipeline_name,

    '05_inventory_analysis' AS stage_name,

    'SUCCESS' AS status,

    CURRENT_TIMESTAMP() AS start_time,

    CURRENT_TIMESTAMP() AS end_time,

    (
        SELECT COUNT(*)
        FROM silver.inventory
    ) AS records_processed,

    0 AS records_failed,

    NULL AS error_message;


-- ============================================================================
-- End of Stage 05 Inventory Analysis
-- ============================================================================